# Reading FITS files

This notebook is used to explore the FITS files located in the `../data` directory.

In [ ]:
from astropy.io import fits
import glob
import os
import pandas as pd
import sys
import getpass
from multiprocessing import BoundedSemaphore, Lock

In [ ]:
# Add hsc_to_lsst to python path to import pipeline
sys.path.append(os.path.abspath("HSC_to_LSST"))
try:
    from hsc_to_lsst.pipeline import query_and_write
    print("Successfully imported query_and_write from hsc_to_lsst.pipeline")
except ImportError as e:
    print(f"Error importing hsc_to_lsst: {e}")
    print("Make sure the HSC_to_LSST directory is in the current directory or python path.")

In [ ]:
# List fits files in contaminants-sample images directory
import os
import glob

data_dir = "../data/contaminants-sample"
fits_dir = os.path.join(data_dir, "contaminants", "cont_50k")
if not os.path.exists(fits_dir):
    fits_dir = os.path.join(data_dir, "cont_50k")
    
fits_files = sorted(glob.glob(os.path.join(fits_dir, "*.fits")))
print(f"Found {len(fits_files)} FITS files in {fits_dir}:")
for f in fits_files[:5]:
    print(os.path.basename(f))


In [ ]:
def explore_fits(file_path):
    print(f"\n{'='*80}\nExploring: {file_path}\n{'='*80}")
    try:
        with fits.open(file_path) as hdul:
            hdul.info()
            
            for i, hdu in enumerate(hdul):
                print(f"\n--- HDU {i} ({hdu.name}) ---")
                display(hdu.header)
                
                if isinstance(hdu, fits.BinTableHDU) or isinstance(hdu, fits.TableHDU):
                    print(f"Table columns: {hdu.columns.names}")
                    # Convert to pandas data frame for nicer display of first few rows
                    try:
                        from astropy.table import Table
                        df = Table(hdu.data).to_pandas()
                        display(df.head())
                        print(f"Table shape: {df.shape}")
                    except Exception as e:
                        print(f"Could not convert to DataFrame: {e}")
                elif isinstance(hdu, fits.ImageHDU) or isinstance(hdu, fits.PrimaryHDU):
                     if hdu.data is not None:
                        print(f"Image data shape: {hdu.data.shape}")
                        print(f"Min: {hdu.data.min()}, Max: {hdu.data.max()}")
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

In [ ]:
# Load contaminants catalog parameters
import os
import pandas as pd
from astropy.table import Table

data_dir = "../data/contaminants-sample"
output_dir = "../data/contaminants-sample-output2"
os.makedirs(output_dir, exist_ok=True)

fits_table_path = os.path.join(data_dir, "table_non_lenses_combined.fits")

# Search for catalog.csv in all possible locations
csv_search_paths = [
    os.path.join(data_dir, "contaminants", "cont_50k", "catalog.csv"),
    os.path.join(data_dir, "cont_50k", "catalog.csv"),
    os.path.join(data_dir, "catalog.csv")
]

csv_table_path = None
for p in csv_search_paths:
    if os.path.exists(p):
        csv_table_path = p
        break

if os.path.exists(fits_table_path):
    print(f"Loading FITS table from {fits_table_path}...")
    tab = Table.read(fits_table_path)
    deep = tab.to_pandas()
elif csv_table_path is not None:
    print(f"Loading CSV table from {csv_table_path}...")
    deep = pd.read_csv(csv_table_path, dtype={'galaxy_id': str})
    out_fits_table = os.path.join(output_dir, "table_non_lenses_combined.fits")
    print(f"Converting CSV to FITS table and saving to {out_fits_table}...")
    tab = Table.from_pandas(deep)
    tab.write(out_fits_table, format='fits', overwrite=True)
else:
    print("Error: No catalog file found!")
    deep = None

if deep is not None:
    print(f"Total contaminants catalog shape: {deep.shape}")
    display(deep.head())


## Query HSC Database

The following cells allow you to query the HSC database for the objects in the DataFrames. You will need to provide your HSC SSP username and password.

In [ ]:
# No HSC credentials needed for local Margherita data
hsc_username = None
hsc_password = None


In [ ]:
# Skipping HSC database queries as we are using the local FITS files from the Margherita sample directly in the following cells.
print("Skipping HSC database queries: using local Margherita FITS files directly.")


## Display Retrieved Images

The following cell displays the images retrieved from the HSC database.

In [ ]:
# Verify and read contaminants FITS files
import os
from astropy.io import fits
try:
    import ipywidgets
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

def verify_and_read_fits(df, base_dir, limit=None):
    print("Reading and verifying contaminants FITS files...")
    count = 0
    missing = 0
    
    fits_dir = os.path.join(base_dir, "contaminants", "cont_50k")
    if not os.path.exists(fits_dir):
        fits_dir = os.path.join(base_dir, "cont_50k")
        
    iterator = df.itertuples()
    total_len = len(df)
    if limit is not None:
        iterator = list(iterator)[:limit]
        total_len = limit
        
    for row in tqdm(iterator, total=total_len, desc="Reading FITS"):
        filename = f"cont_{row.galaxy_id}_bcg{row.cont_has_bcg}_type{row.cont_type}.fits"
        sim_path = os.path.join(fits_dir, filename)
        
        if not os.path.exists(sim_path):
            missing += 1
            continue
            
        try:
            with fits.open(sim_path) as hdul:
                _ = hdul[0].data.shape
            count += 1
        except Exception as e:
            print(f"Error reading FITS for galaxy {row.galaxy_id}: {e}")
            
    print(f"Finished reading FITS: {count} valid files read, {missing} files missing on disk.")

final_sample_dir = "../data/contaminants-sample"
if deep is not None:
    print("\n--- Verifying Contaminants FITS Images ---")
    verify_and_read_fits(deep, final_sample_dir, limit=None)


## Apply LSST Degradation

This section applies the `read_degrade_hsc` function to the retrieved HSC images to simulate LSST-like images.
It then displays the original HSC images side-by-side with the degraded LSST images.

### Note on Image Sizes and PSF

- **Image Sizes**: The HSC images are approximately 20 arcseconds on a side (119x119 pixels at 0.168"/pix). The LSST images are 8.2 arcseconds on a side (41x41 pixels at 0.2"/pix). In the comparison plots below, the **HSC images are zoomed in** to match the field of view of the LSST images (8.2" x 8.2") to allow for a direct comparison of features at the same angular size.
- **PSF Model**: The LSST PSF is modeled as a Gaussian kernel. The FWHM (Full Width at Half Maximum) of this Gaussian is sampled from LSST DP0.2 simulations, which provide realistic seeing conditions expected for the LSST survey.

In [ ]:
from hsc_to_lsst.pipeline import degrade_images, zero_point_change
from hsc_to_lsst.lsst_props import dp0_gmm_sampler
from hsc_to_lsst.pipeline import HSC_MEAN_FWHM, LSST_SINGLE_EXP_AVG_ZERO_POINTS
from hsc_to_lsst.data_degradation.hsc_degradation import hsc_to_lsst
from multiprocessing import Lock
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
try:
    import ipywidgets
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

def display_comparison_example(orig_cube, degraded_cube, galaxy_id):
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    bands = ['g', 'r', 'i', 'z', 'y']
    
    def _norm(arr):
        p1, p99 = np.nanpercentile(arr, [1, 99])
        denom = p99 - p1 if p99 > p1 else 1.0
        return np.clip((arr - p1) / denom, 0, 1)
        
    for i, band in enumerate(bands):
        axes[0, i].imshow(_norm(orig_cube[i]), origin='lower', cmap='gray')
        axes[0, i].set_title(f"Original {band}")
        axes[0, i].axis("off")
        
        axes[1, i].imshow(_norm(degraded_cube[i]), origin='lower', cmap='gray')
        axes[1, i].set_title(f"LSST {band}")
        axes[1, i].axis("off")
        
    plt.suptitle(f"Comparison for Galaxy {galaxy_id}", fontsize=14)
    plt.tight_layout()
    plt.show()

def perform_degradation_contaminants(df, base_dir, output_dir, coadd_years=1, limit=None, show_examples_count=2):
    print(f"Degrading contaminants images to {output_dir}...")
    out_fits_dir = os.path.join(output_dir, "cont_50k")
    os.makedirs(out_fits_dir, exist_ok=True)
    
    fits_dir = os.path.join(base_dir, "contaminants", "cont_50k")
    if not os.path.exists(fits_dir):
        fits_dir = os.path.join(base_dir, "cont_50k")
        
    dp0_sampler = dp0_gmm_sampler(str(coadd_years))
    log_lock = Lock()
    log_file = os.path.join(output_dir, "degrade.log")
    
    count = 0
    shown_count = 0
    
    iterator = df.itertuples()
    total_len = len(df)
    if limit is not None:
        iterator = list(iterator)[:limit]
        total_len = limit
        
    for row in tqdm(iterator, total=total_len, desc="Degrading"):
        filename = f"cont_{row.galaxy_id}_bcg{row.cont_has_bcg}_type{row.cont_type}.fits"
        sim_path = os.path.join(fits_dir, filename)
        
        if not os.path.exists(sim_path):
            continue
            
        try:
            with fits.open(sim_path) as hdul:
                cube = hdul[0].data.astype(np.float32)
                
            images = [cube[i] for i in range(5)]
            psfs = [None] * 5
            
            success, degraded_images, mag_change = degrade_images(
                images,
                psfs,
                dp0_sampler,
                hsc_pix_scale=0.168,
                zp_rms_frac_thresh=0.3,
                zp_max_mag_change=2.0,
                lsst_size_pix=41,
                bands='grizy',
                log_file=log_file,
                log_lock=log_lock,
                log_prefix=f"{row.galaxy_id}: "
            )
            
            if success:
                out_path = os.path.join(out_fits_dir, filename)
                degraded_cube = np.stack(degraded_images, axis=0)
                
                hdu = fits.PrimaryHDU(data=degraded_cube)
                hdu.writeto(out_path, overwrite=True)
                
                if shown_count < show_examples_count:
                    display_comparison_example(cube, degraded_cube, row.galaxy_id)
                    shown_count += 1
        except Exception as e:
            pass
            
        count += 1
    print(f"Degradation completed for {count} galaxies.")

final_sample_dir = "../data/contaminants-sample"
degraded_dir = "../data/contaminants-sample-output2"
if deep is not None:
    print("\n--- Degrading Contaminants Data ---")
    perform_degradation_contaminants(deep, final_sample_dir, degraded_dir, limit=None, show_examples_count=2)
